In [ ]:
"""
GO Ontology Supervised Fine-Tuning (GOSFT) Dataset Generation Pipeline
======================================================================

This notebook generates a comprehensive Question-Answering dataset for Gene Ontology (GO)
terms to enable supervised fine-tuning of large language models on GO knowledge.

Pipeline Overview:
1. Parse GO OBO file (go-basic.obo)
2. Build complete tree structures for each GO category (BP, MF, CC)
3. Find all root-to-leaf paths through the ontology
4. Generate 7 types of Q&A pairs per GO term
5. Clean and format answers using natural language response templates
6. Export dataset and upload to HuggingFace

Question Types Generated (7 per GO term):
- Category: Which GO category does this term belong to?
- Name: What is the official name of this GO term?
- Definition: What is the scientific definition?
- Parents: What are the parent terms?
- Children: What are the child terms?
- Siblings: What are sibling terms at the same level?
- Paths: What are the complete hierarchical paths?

Output:
- Category JSON files (cc, mf, bp analysis)
- Combined GO analysis JSON
- Final QA dataset in Parquet format (~300K Q&A pairs)


NOTE: Clone repository first. Requires go-basic.obo file in working directory.
"""

In [ ]:
"""
Install Required Packages
-------------------------
Install necessary Python packages for GO ontology processing.
"""

import sys

print("Installing required packages...")
!pip install -q datasets huggingface_hub pyyaml networkx matplotlib

print("Package installation complete")

In [ ]:
"""
Import Libraries and Configure Environment
------------------------------------------
"""

import json
import random
import pandas as pd
import re
from typing import Dict, List, Any
import os
from collections import defaultdict, deque
from multiprocessing import Pool
from functools import partial
import warnings

# Set random seed for reproducibility
random.seed(42)

# Suppress warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully")
print("Random seed set to 42 for reproducibility")

In [ ]:
"""
Configuration Parameters
------------------------
Set paths and parameters for the pipeline.
"""

# File paths
OBO_FILE = "go-basic.obo"
OUTPUT_DIR = "go_analysis_output"

# Processing parameters
N_CORES = 90  # Number of CPU cores for multiprocessing
BATCH_SIZE = 500  # Batch size for path assignment

# GO category roots
GO_ROOTS = {
    'CC': ('cellular_component', 'GO:0005575'),
    'MF': ('molecular_function', 'GO:0003674'),
    'BP': ('biological_process', 'GO:0008150')
}

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Configuration:")
print(f"  OBO file: {OBO_FILE}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  CPU cores: {N_CORES}")
print(f"  Batch size: {BATCH_SIZE}")

In [ ]:
"""
OBO File Parsing Functions
--------------------------
"""

def parse_obo_file(filename):
    """Parse GO OBO file and extract all terms with their properties."""
    print(f"Parsing OBO file: {filename}")

    with open(filename, 'r', encoding='utf-8') as f:
        content = f.read()

    print(f"File loaded: {len(content):,} characters")

    # Parse metadata
    metadata = {}
    lines = content.split('\n')

    for line in lines[:50]:
        if line.startswith('format-version:'):
            metadata['format_version'] = line.split(':', 1)[1].strip()
        elif line.startswith('data-version:'):
            metadata['data_version'] = line.split(':', 1)[1].strip()
        elif line.startswith('date:'):
            metadata['date'] = line.split(':', 1)[1].strip()

    # Parse GO terms
    go_terms = {}
    current_term = None

    for line in lines:
        line = line.strip()

        if line == '[Term]':
            if current_term and 'id' in current_term:
                go_terms[current_term['id']] = current_term
            current_term = {
                'parents': [],
                'part_of_parents': [],
                'relationships': []
            }
        elif current_term is not None:
            if line.startswith('id:'):
                current_term['id'] = line.split(':', 1)[1].strip()
            elif line.startswith('name:'):
                current_term['name'] = line.split(':', 1)[1].strip()
            elif line.startswith('namespace:'):
                current_term['namespace'] = line.split(':', 1)[1].strip()
            elif line.startswith('def:'):
                def_match = re.match(r'def:\s*"([^"]*)"', line)
                if def_match:
                    current_term['definition'] = def_match.group(1)
            elif line.startswith('is_a:'):
                parent_id = line.split('!')[0].split(':', 1)[1].strip()
                current_term['parents'].append(parent_id)
            elif line.startswith('relationship: part_of'):
                parent_id = line.split('!')[0].split()[-1].strip()
                current_term['part_of_parents'].append(parent_id)
            elif line.startswith('is_obsolete:'):
                current_term['is_obsolete'] = line.split(':', 1)[1].strip().lower() == 'true'
            elif line.startswith('synonym:'):
                if 'synonyms' not in current_term:
                    current_term['synonyms'] = []
                syn_match = re.match(r'synonym:\s*"([^"]*)"', line)
                if syn_match:
                    current_term['synonyms'].append(syn_match.group(1))

    if current_term and 'id' in current_term:
        go_terms[current_term['id']] = current_term

    print(f"Parsed {len(go_terms):,} GO terms")
    print(f"Metadata: {metadata}")

    return {
        'metadata': metadata,
        'go_terms': go_terms
    }

print("OBO parsing function defined")

In [ ]:
"""
Tree Analysis Functions - Part 1
--------------------------------
"""

def build_category_tree_structure(category_terms, root_id):
    """Build tree structure with children, parents, siblings, depth."""
    print("Building tree structure...")

    # Initialize children lists
    for go_id, term in category_terms.items():
        term['is_a_children'] = []
        term['part_of_children'] = []
        term['all_children'] = []
        term['all_parents'] = list(set(term.get('parents', []) + term.get('part_of_parents', [])))

    # Build children relationships
    for go_id, term in category_terms.items():
        for parent_id in term.get('parents', []):
            if parent_id in category_terms:
                category_terms[parent_id]['is_a_children'].append(go_id)

        for parent_id in term.get('part_of_parents', []):
            if parent_id in category_terms:
                category_terms[parent_id]['part_of_children'].append(go_id)

    # Combine all children
    for go_id, term in category_terms.items():
        term['all_children'] = list(set(term['is_a_children'] + term['part_of_children']))

    # Calculate depths using BFS
    print("Calculating depths...")
    depths = {root_id: 0}
    queue = deque([root_id])

    while queue:
        current_id = queue.popleft()
        current_depth = depths[current_id]

        if current_id in category_terms:
            for child_id in category_terms[current_id]['all_children']:
                if child_id not in depths:
                    depths[child_id] = current_depth + 1
                    queue.append(child_id)

    # Add depth and leaf info
    for go_id, term in category_terms.items():
        term['depth'] = depths.get(go_id, -1)
        term['is_leaf'] = len(term['all_children']) == 0

    # Find siblings
    print("Finding siblings...")
    for go_id, term in category_terms.items():
        siblings = set()
        for parent_id in term['all_parents']:
            if parent_id in category_terms:
                for sibling_id in category_terms[parent_id]['all_children']:
                    if sibling_id != go_id:
                        siblings.add(sibling_id)
        term['siblings'] = list(siblings)

    print("Tree structure built")
    return category_terms

print("Tree structure building function defined")

In [ ]:
"""
Tree Analysis Functions - Part 2
--------------------------------
"""

# Global variable for multiprocessing
ALL_COMPLETE_PATHS = []

def assign_paths_to_term_worker(go_id):
    """Worker function to find all complete paths through a GO term."""
    paths_through_term = []
    for path in ALL_COMPLETE_PATHS:
        if go_id in path:
            paths_through_term.append(path)
    return go_id, paths_through_term

def find_all_complete_paths(category_terms, root_id, n_cores=90):
    """Find all complete paths from root to leaves."""
    global ALL_COMPLETE_PATHS

    print(f"Finding ALL complete root-to-leaf paths...")
    print(f"Root: {root_id}")

    def find_all_root_to_leaf_paths():
        """Find every possible path from root to every leaf using DFS"""
        paths = []

        def dfs_to_leaves(current_id, path, visited):
            if current_id in visited:
                return

            visited.add(current_id)

            if current_id in category_terms:
                children = category_terms[current_id]['all_children']

                if not children:
                    paths.append(path)
                else:
                    for child_id in children:
                        dfs_to_leaves(child_id, path + [child_id], visited.copy())

        dfs_to_leaves(root_id, [root_id], set())
        return paths

    print("Finding all root-to-leaf paths...")
    all_complete_paths = find_all_root_to_leaf_paths()
    print(f"Found {len(all_complete_paths):,} complete paths")

    ALL_COMPLETE_PATHS = all_complete_paths

    print(f"Assigning paths to GO terms...")
    go_ids = list(category_terms.keys())
    print(f"Processing {len(go_ids)} terms with {n_cores} cores...")

    all_term_paths = {}

    batch_size = BATCH_SIZE
    total_batches = (len(go_ids) + batch_size - 1) // batch_size

    for batch_idx in range(total_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(go_ids))
        batch_go_ids = go_ids[start_idx:end_idx]

        print(f"Processing batch {batch_idx + 1}/{total_batches} ({len(batch_go_ids)} terms)")

        with Pool(n_cores) as pool:
            results = pool.map(assign_paths_to_term_worker, batch_go_ids)

        for go_id, paths in results:
            all_term_paths[go_id] = paths

    print(f"Path assignment complete!")
    return all_term_paths, all_complete_paths

print("Path finding functions defined")

In [ ]:
"""
Category Processing Functions
-----------------------------
"""

def create_final_category_json(category_terms, all_term_paths, category_code):
    """Create final JSON structure for a category."""
    print("Creating final JSON structure...")

    category_analysis = {}

    for go_id, term in category_terms.items():
        category_analysis[go_id] = {
            'id': go_id,
            'name': term.get('name', ''),
            'definition': term.get('definition', ''),
            'category': category_code,
            'parents': term['all_parents'],
            'children': term['all_children'],
            'siblings': term['siblings'],
            'all_paths_through_term': all_term_paths.get(go_id, []),
            'depth': term['depth'],
            'is_leaf': term['is_leaf'],
            'relationship_types': {
                'is_a_parents': term.get('parents', []),
                'part_of_parents': term.get('part_of_parents', []),
                'is_a_children': term['is_a_children'],
                'part_of_children': term['part_of_children']
            }
        }

    return category_analysis

def process_single_category(go_terms, category_code, namespace, root_id, n_cores=90):
    """Process a single GO category."""
    print(f"\n{'='*80}")
    print(f"PROCESSING {category_code} ({namespace})")
    print(f"{'='*80}")

    category_terms = {go_id: term for go_id, term in go_terms.items()
                     if term.get('namespace') == namespace and not term.get('is_obsolete', False)}

    print(f"Processing {len(category_terms):,} {category_code} terms")

    if len(category_terms) == 0:
        print(f"No terms found for {category_code}")
        return None, None

    category_terms = build_category_tree_structure(category_terms, root_id)
    all_term_paths, all_complete_paths = find_all_complete_paths(category_terms, root_id, n_cores)
    category_analysis = create_final_category_json(category_terms, all_term_paths, category_code)

    output_file = os.path.join(OUTPUT_DIR, f'{category_code.lower()}_complete_paths_analysis.json')
    with open(output_file, 'w') as f:
        json.dump(category_analysis, f, indent=2)

    print(f"Saved {category_code} analysis to {output_file}")

    total_terms = len(category_analysis)
    leaf_terms = sum(1 for term in category_analysis.values() if term['is_leaf'])
    max_depth = max(term['depth'] for term in category_analysis.values()) if category_analysis else 0

    print(f"\n{category_code} STATS:")
    print(f"  Total terms: {total_terms:,}")
    print(f"  Leaf terms: {leaf_terms:,} ({leaf_terms/total_terms*100:.1f}%)")
    print(f"  Maximum depth: {max_depth}")
    print(f"  Total paths: {len(all_complete_paths):,}")

    return category_analysis, all_complete_paths

def process_all_go_categories(go_terms, n_cores=90):
    """Process all GO categories."""
    print("PROCESSING ALL GO CATEGORIES")
    print("="*80)

    all_results = {}
    all_paths_data = {}

    for category_code, (namespace, root_id) in GO_ROOTS.items():
        category_analysis, all_complete_paths = process_single_category(
            go_terms, category_code, namespace, root_id, n_cores
        )

        if category_analysis:
            all_results[category_code] = category_analysis
            all_paths_data[category_code] = all_complete_paths

    combined_output_file = os.path.join(OUTPUT_DIR, 'all_go_categories_complete_paths.json')
    with open(combined_output_file, 'w') as f:
        json.dump(all_results, f, indent=2)

    print(f"\nSaved combined analysis to {combined_output_file}")
    print(f"\nALL CATEGORIES PROCESSING COMPLETE!")

    return all_results, all_paths_data

print("Category processing functions defined")

In [ ]:
"""
Execute OBO Parsing and Tree Analysis
-------------------------------------
"""

print("="*80)
print("STEP 1: PARSE OBO FILE AND BUILD TREE STRUCTURES")
print("="*80)

result = parse_obo_file(OBO_FILE)
go_terms = result['go_terms']
metadata = result['metadata']

print(f"\nOBO file metadata:")
for key, value in metadata.items():
    print(f"  {key}: {value}")

all_go_analysis, all_paths_data = process_all_go_categories(go_terms, n_cores=N_CORES)

print(f"\nFiles generated in {OUTPUT_DIR}:")
print(f"  - cc_complete_paths_analysis.json")
print(f"  - mf_complete_paths_analysis.json")
print(f"  - bp_complete_paths_analysis.json")
print(f"  - all_go_categories_complete_paths.json")
print("\n" + "="*80)
print("TREE ANALYSIS COMPLETE")
print("="*80)

In [ ]:
"""
Response Templates
-----------------
Natural language response templates for answers.
"""

# Response templates for when data is available
RESPONSE_TEMPLATES_AVAILABLE = {
    "category": [
        "The GO category of {go_id} is {category}.",
        "{go_id} belongs to the {category} category.",
        "{go_id} is classified as {category}.",
        "The ontology branch for {go_id} is {category}.",
        "{go_id} is categorized under {category}."
    ],
    "name": [
        "The name of {go_id} is {name}.",
        "{go_id} is called {name}.",
        "The GO term name for {go_id} is {name}.",
        "{go_id} is named {name}.",
        "The official name of {go_id} is {name}."
    ],
    "definition": [
        "The definition of {go_id} is: {definition}",
        "{go_id} is defined as: {definition}",
        "{go_id} means: {definition}",
        "The scientific definition of {go_id} is: {definition}",
        "{go_id} is described as: {definition}"
    ],
    "parents": [
        "The parents of {go_id} are: {parents_list}.",
        "The parent terms of {go_id} are: {parents_list}.",
        "{go_id} has the following parents: {parents_list}.",
        "The direct parents of {go_id} are: {parents_list}.",
        "{go_id} descends from: {parents_list}."
    ],
    "children": [
        "The children of {go_id} are: {children_list}.",
        "The child terms of {go_id} are: {children_list}.",
        "{go_id} has the following children: {children_list}.",
        "The direct children of {go_id} are: {children_list}.",
        "{go_id} is a parent of: {children_list}."
    ],
    "siblings": [
        "The siblings of {go_id} are: {siblings_list}.",
        "The sibling terms of {go_id} are: {siblings_list}.",
        "{go_id} has the following siblings: {siblings_list}.",
        "Terms that share parents with {go_id} are: {siblings_list}.",
        "The sibling nodes of {go_id} are: {siblings_list}."
    ],
    "paths": [
        "The complete paths through {go_id} are: {paths_list}.",
        "All paths containing {go_id} are: {paths_list}.",
        "{go_id} appears in the following paths: {paths_list}.",
        "The root-to-leaf paths for {go_id} are: {paths_list}.",
        "The hierarchical paths through {go_id} are: {paths_list}."
    ]
}

# Response templates for when data is not available
RESPONSE_TEMPLATES_NOT_AVAILABLE = {
    "category": [
        "{go_id} has no category information.",
        "No category found for {go_id}.",
        "The category for {go_id} is not available.",
        "Category information for {go_id} is missing.",
        "{go_id} does not have a defined category."
    ],
    "name": [
        "{go_id} has no name information.",
        "No name found for {go_id}.",
        "The name for {go_id} is not available.",
        "Name information for {go_id} is missing.",
        "{go_id} does not have a defined name."
    ],
    "definition": [
        "{go_id} has no definition available.",
        "No definition found for {go_id}.",
        "The definition for {go_id} is not available.",
        "Definition information for {go_id} is missing.",
        "{go_id} does not have a defined description."
    ],
    "parents": [
        "{go_id} has no parents.",
        "There are no parent terms for {go_id}.",
        "{go_id} is a root node with no parents.",
        "No parents found for {go_id}.",
        "{go_id} does not have any parent terms."
    ],
    "children": [
        "{go_id} has no children.",
        "There are no child terms for {go_id}.",
        "{go_id} is a leaf node with no children.",
        "No children found for {go_id}.",
        "{go_id} does not have any child terms."
    ],
    "siblings": [
        "{go_id} has no siblings.",
        "There are no sibling terms for {go_id}.",
        "{go_id} is an only child with no siblings.",
        "No siblings found for {go_id}.",
        "{go_id} does not have any sibling terms."
    ],
    "paths": [
        "{go_id} has no paths available.",
        "No paths found for {go_id}.",
        "There are no complete paths containing {go_id}.",
        "Path information for {go_id} is not available.",
        "{go_id} does not appear in any complete paths."
    ]
}

print("Response templates defined")
print(f"Available templates: {len(RESPONSE_TEMPLATES_AVAILABLE)} types")
print(f"Not available templates: {len(RESPONSE_TEMPLATES_NOT_AVAILABLE)} types")

In [ ]:
"""
QA Generation Functions - Part 1
--------------------------------
"""

def load_go_data():
    """Load GO analysis data from JSON files"""
    print("Loading GO analysis data...")

    with open(os.path.join(OUTPUT_DIR, 'cc_complete_paths_analysis.json'), 'r') as f:
        cc_data = json.load(f)

    with open(os.path.join(OUTPUT_DIR, 'mf_complete_paths_analysis.json'), 'r') as f:
        mf_data = json.load(f)

    with open(os.path.join(OUTPUT_DIR, 'bp_complete_paths_analysis.json'), 'r') as f:
        bp_data = json.load(f)

    print(f"Loaded: CC({len(cc_data)}), MF({len(mf_data)}), BP({len(bp_data)})")

    return {
        'CC': cc_data,
        'MF': mf_data,
        'BP': bp_data
    }

def clean_definition(definition):
    """Remove citations from definition text"""
    if not definition:
        return ""

    cleaned = re.sub(r'\[PMID:\d+\]', '', definition)
    cleaned = re.sub(r'\[GOC:[^\]]+\]', '', cleaned)
    cleaned = re.sub(r'\[ISBN:[^\]]+\]', '', cleaned)
    cleaned = re.sub(r'\[[A-Z]+:[^\]]+\]', '', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)

    return cleaned.strip()

def format_path_readable(path):
    """Format a path as readable arrow structure"""
    if not path or len(path) == 0:
        return ""

    result = path[0]
    for term in path[1:]:
        result = f"{result} → {term}"

    return result

def format_answer(answer_data, question_type, go_id, used_response_templates):
    """
    Format answers using natural language response templates.

    Args:
        answer_data: Raw answer data
        question_type: Type of question
        go_id: GO term ID
        used_response_templates: Dictionary tracking used templates

    Returns:
        Formatted natural language answer string
    """
    # Initialize tracking for this GO term
    if go_id not in used_response_templates:
        used_response_templates[go_id] = {}
    if question_type not in used_response_templates[go_id]:
        used_response_templates[go_id][question_type] = 0

    # Determine if data is available
    data_available = False

    if question_type == 'category':
        data_available = bool(answer_data)
    elif question_type == 'name':
        data_available = bool(answer_data)
    elif question_type == 'definition':
        cleaned = clean_definition(answer_data)
        data_available = bool(cleaned)
        answer_data = cleaned
    elif question_type in ['parents', 'children', 'siblings']:
        data_available = isinstance(answer_data, list) and len(answer_data) > 0
    elif question_type == 'paths':
        data_available = isinstance(answer_data, list) and len(answer_data) > 0

    # Select template set
    if data_available:
        templates = RESPONSE_TEMPLATES_AVAILABLE[question_type]
    else:
        templates = RESPONSE_TEMPLATES_NOT_AVAILABLE[question_type]

    # Select specific template
    template_idx = used_response_templates[go_id][question_type] % len(templates)
    template = templates[template_idx]
    used_response_templates[go_id][question_type] += 1

    # Format answer based on type
    if not data_available:
        # No data available - use not available template
        return template.format(go_id=go_id)

    # Data available - format based on type
    if question_type == 'category':
        category_map = {
            'CC': 'Cellular Component',
            'MF': 'Molecular Function',
            'BP': 'Biological Process'
        }
        category = category_map.get(answer_data, str(answer_data))
        return template.format(go_id=go_id, category=category)

    elif question_type == 'name':
        return template.format(go_id=go_id, name=answer_data)

    elif question_type == 'definition':
        return template.format(go_id=go_id, definition=answer_data)

    elif question_type in ['parents', 'children', 'siblings']:
        list_str = "; ".join(answer_data)
        key_name = f"{question_type}_list"
        return template.format(go_id=go_id, **{key_name: list_str})

    elif question_type == 'paths':
        formatted_paths = [format_path_readable(path) for path in answer_data[:10]]
        paths_str = " | ".join(formatted_paths)
        return template.format(go_id=go_id, paths_list=paths_str)

    return str(answer_data)

print("Data loading and answer formatting functions defined")

In [ ]:
"""
QA Generation Functions - Part 2
--------------------------------
"""

def generate_qa_for_go_term(go_id, go_data, used_templates, used_response_templates):
    """Generate 7 Q&A pairs for a single GO term."""
    qa_pairs = []

    question_categories = ['category', 'name', 'definition', 'parents', 'children', 'siblings', 'paths']

    for q_category in question_categories:
        # Select question template
        if go_id not in used_templates:
            used_templates[go_id] = {}

        if q_category not in used_templates[go_id]:
            used_templates[go_id][q_category] = 0

        template_idx = used_templates[go_id][q_category] % len(QUESTION_TEMPLATES[q_category])
        template = QUESTION_TEMPLATES[q_category][template_idx]
        used_templates[go_id][q_category] += 1

        # Generate question
        question = template.format(go_id=go_id)

        # Get answer data
        if q_category == 'category':
            answer_data = go_data.get('category', '')
        elif q_category == 'name':
            answer_data = go_data.get('name', '')
        elif q_category == 'definition':
            answer_data = go_data.get('definition', '')
        elif q_category == 'parents':
            answer_data = go_data.get('parents', [])
        elif q_category == 'children':
            answer_data = go_data.get('children', [])
        elif q_category == 'siblings':
            answer_data = go_data.get('siblings', [])
        elif q_category == 'paths':
            answer_data = go_data.get('all_paths_through_term', [])
        else:
            answer_data = ""

        # Format answer with natural language templates
        answer = format_answer(answer_data, q_category, go_id, used_response_templates)

        # Create Q&A pair
        qa_pair = {
            'question': question,
            'answer': answer,
            'go_id': go_id,
            'go_name': go_data.get('name', ''),
            'go_category': go_data.get('category', '')
        }

        qa_pairs.append(qa_pair)

    return qa_pairs

def generate_all_qa_pairs(go_analysis_data):
    """Generate all Q&A pairs for all GO terms."""
    print("Generating Q&A pairs for all GO terms...")

    all_qa_pairs = []
    used_templates = {}
    used_response_templates = {}

    total_terms = sum(len(category_data) for category_data in go_analysis_data.values())
    processed = 0

    for category_code, category_data in go_analysis_data.items():
        print(f"\nProcessing {category_code} ({len(category_data):,} terms)...")

        for go_id, go_data in category_data.items():
            qa_pairs = generate_qa_for_go_term(go_id, go_data, used_templates, used_response_templates)
            all_qa_pairs.extend(qa_pairs)

            processed += 1
            if processed % 1000 == 0:
                print(f"  Processed {processed:,}/{total_terms:,} terms ({processed/total_terms*100:.1f}%)")

    print(f"\nGenerated {len(all_qa_pairs):,} Q&A pairs from {total_terms:,} GO terms")
    print(f"Average: {len(all_qa_pairs)/total_terms:.1f} Q&A pairs per term")

    return all_qa_pairs

print("Q&A generation functions defined")

In [ ]:
"""
Execute QA Generation
--------------------
"""

print("="*80)
print("STEP 2: GENERATE Q&A PAIRS")
print("="*80)

go_analysis_data = load_go_data()
all_qa_pairs = generate_all_qa_pairs(go_analysis_data)

print(f"\n{'='*80}")
print("QA GENERATION STATISTICS")
print("="*80)

# Count by GO category
category_counts = {}
for qa in all_qa_pairs:
    cat = qa['go_category']
    category_counts[cat] = category_counts.get(cat, 0) + 1

print("\nQuestions by GO category:")
for cat, count in sorted(category_counts.items()):
    print(f"  {cat}: {count:,} ({count/len(all_qa_pairs)*100:.1f}%)")

# Sample Q&A pairs
print("\nSample Q&A pairs (first 3):")
for i, qa in enumerate(all_qa_pairs[:3], 1):
    print(f"\n{i}. Question: {qa['question']}")
    answer_preview = qa['answer'][:150] + "..." if len(qa['answer']) > 150 else qa['answer']
    print(f"   Answer: {answer_preview}")
    print(f"   GO ID: {qa['go_id']}, Category: {qa['go_category']}")

print("\n" + "="*80)
print("QA GENERATION COMPLETE")
print("="*80)

In [ ]:
"""
Save QA Dataset
--------------
"""

print("="*80)
print("STEP 3: SAVE QA DATASET")
print("="*80)

df_qa = pd.DataFrame(all_qa_pairs)

print(f"\nDataset shape: {df_qa.shape}")
print(f"Columns: {list(df_qa.columns)}")

output_parquet = os.path.join(OUTPUT_DIR, "go_qa_dataset.parquet")
df_qa.to_parquet(output_parquet, index=False)

file_size = os.path.getsize(output_parquet) / (1024 * 1024)
print(f"\nSaved dataset to: {output_parquet}")
print(f"File size: {file_size:.1f} MB")

df_verify = pd.read_parquet(output_parquet)
print(f"\nVerification:")
print(f"  Loaded shape: {df_verify.shape}")
print(f"  Matches original: {df_qa.shape == df_verify.shape}")

print(f"\nSample from saved dataset:")
print(df_verify.head(3))

print("\n" + "="*80)
print("DATASET SAVED SUCCESSFULLY")
print("="*80)